# Four LoRA methods on SmolLM2-135M

Self-contained Colab experiment for LoRATune. It is **not** the CSC2516 homework notebook. It reuses the same model, dataset, chat template, next-token loss, and ten prompts so the four PEFT recipes can be compared on one GPU run.

| Method | What changes | Config |
| --- | --- | --- |
| `original_lora` | baseline adapter | attention only, \(r=8\), \(\alpha=16\), scale \(\alpha/r\), float base |
| `qlora` | frozen base storage | same adapter as above, 4-bit NF4 base |
| `all_linear` | which layers + rank | all linear layers, \(r=16\), \(\alpha=16\), scale \(\alpha/r\) |
| `rslora` | scale only | same as `all_linear`, scale \(\alpha/\sqrt{r}\) |

**Colab setup**
1. Runtime → Change runtime type → **T4 GPU** (or any CUDA GPU).
2. Run every cell top to bottom.
3. Artifacts land in Google Drive under `MyDrive/loratune_experiments/four_methods/`.
4. After the run finishes, download that folder and tell the local agent training succeeded so the report and package refactor can be filled from those files.

Expected wall time on a T4: roughly 1–2 hours for four methods × 3 epochs on 2,260 chats.


## 0. Install and mount Drive


In [ ]:
# GPU required for QLoRA (bitsandbytes). Check before spending an hour training.
!nvidia-smi

!pip -q install -U transformers datasets accelerate peft bitsandbytes matplotlib pandas tqdm


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/loratune_experiments/four_methods")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "adapters").mkdir(exist_ok=True)
(DRIVE_ROOT / "figures").mkdir(exist_ok=True)
(DRIVE_ROOT / "generations").mkdir(exist_ok=True)
print("Saving results to:", DRIVE_ROOT)


## 1. Shared setup

Same model, SmolTalk split, chat template, and prompts as the homework / LoRATune page.


In [ ]:
import json
import math
import os
import time
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)

assert torch.cuda.is_available(), "Enable a CUDA GPU runtime before continuing."
DEVICE = "cuda"
torch.manual_seed(42)
np.random.seed(42)

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
DATASET_NAME = "HuggingFaceTB/smoltalk"
DATASET_CONFIG = "everyday-conversations"

LEARNING_RATE = 1e-4
EPOCHS = 3
BATCH_SIZE = 1
MAX_LENGTH = 512
MAX_NEW_TOKENS = 50

ATTENTION_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]
ALL_LINEAR_TARGETS = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

METHODS = {
    "original_lora": {
        "label": "Original LoRA",
        "r": 8,
        "alpha": 16,
        "target_modules": ATTENTION_TARGETS,
        "use_rslora": False,
        "load_in_4bit": False,
        "what_changed": "attention only, r=8, scale alpha/r, float base",
    },
    "qlora": {
        "label": "QLoRA",
        "r": 8,
        "alpha": 16,
        "target_modules": ATTENTION_TARGETS,
        "use_rslora": False,
        "load_in_4bit": True,
        "what_changed": "same adapter as original, 4-bit NF4 frozen base",
    },
    "all_linear": {
        "label": "LoRA on all linear layers",
        "r": 16,
        "alpha": 16,
        "target_modules": ALL_LINEAR_TARGETS,
        "use_rslora": False,
        "load_in_4bit": False,
        "what_changed": "all linear layers, r=16, scale alpha/r",
    },
    "rslora": {
        "label": "rsLoRA",
        "r": 16,
        "alpha": 16,
        "target_modules": ALL_LINEAR_TARGETS,
        "use_rslora": True,
        "load_in_4bit": False,
        "what_changed": "same layers/rank as all_linear, scale alpha/sqrt(r)",
    },
}

print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:
def apply_chat_template(messages: List) -> str:
    lines = []
    for message in messages:
        role = message["role"]
        content = message["content"]
        if role == "user":
            lines.append(f"<|im_start|><|user|>: {content}<|im_end|>\n")
        elif role == "assistant":
            lines.append(f"<|im_start|><|assistant|>: {content}<|im_end|>\n")
    return "".join(lines)


def assistant_prompt(messages: List) -> str:
    return apply_chat_template(messages) + "\n<|im_start|><|assistant|>: "


PROMPTS = [
    [{"role": "user", "content": "Explain photosynthesis in one sentence."}],
    [{"role": "user", "content": "Give me a simple pasta recipe for two people."}],
    [{"role": "user", "content": "How do I apologize to a friend after forgetting their birthday?"}],
    [{"role": "user", "content": "What should I pack for a weekend hike?"}],
    [{"role": "user", "content": "Recommend a movie if I liked Inception."}],
    [{"role": "user", "content": "Help me write a short out-of-office email."}],
    [{"role": "user", "content": "What is a good way to fall asleep faster?"}],
    [{"role": "user", "content": "Explain the offside rule in soccer."}],
    [{"role": "user", "content": "Suggest a weeknight dinner that uses chicken and rice."}],
    [{"role": "user", "content": "How can I politely decline a party invitation?"}],
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = "<|pad_token|>"
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(dataset)
print("train rows:", len(dataset["train"]), "test rows:", len(dataset["test"]))


In [ ]:
def tokenize_messages(example):
    text = apply_chat_template(example["messages"])
    tokens = tokenizer(text, truncation=True, max_length=MAX_LENGTH)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


train_data = dataset["train"].map(
    tokenize_messages,
    remove_columns=dataset["train"].column_names,
)
train_data.set_format("torch")
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

print("Training batches per epoch:", len(train_loader))


## 2. Model builders, training, and evaluation

`original_lora` and `qlora` share the same adapter shape. `all_linear` and `rslora` share the wider target list. Only one knob changes inside each planned comparison.


In [ ]:
def count_trainable_parameters(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def model_device(model):
    return next(p for p in model.parameters() if p.device.type != "meta").device


def estimate_frozen_base_memory_mb(model, load_in_4bit: bool) -> float:
    # Rough weight storage only: 4-bit ≈ 0.5 byte/param, float32 ≈ 4 bytes/param.
    total = sum(p.numel() for p in model.parameters())
    bytes_per = 0.5 if load_in_4bit else 4.0
    return total * bytes_per / (1024 ** 2)


def build_model(method_cfg: Dict):
    if method_cfg["load_in_4bit"]:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb,
            device_map={"": 0},
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
        )
        model.to(DEVICE)

    lora_config = LoraConfig(
        r=method_cfg["r"],
        lora_alpha=method_cfg["alpha"],
        target_modules=method_cfg["target_modules"],
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        use_rslora=method_cfg["use_rslora"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


@torch.no_grad()
def validation_loss(model) -> float:
    model.eval()
    device = model_device(model)
    losses = []
    for row in tqdm(dataset["test"], desc="test loss", leave=False):
        text = apply_chat_template(row["messages"])
        batch = tokenizer(text, return_tensors="pt")
        batch["labels"] = batch["input_ids"].clone()
        batch = {k: v.to(device) for k, v in batch.items()}
        losses.append(float(model(**batch).loss.item()))
    return float(np.mean(losses))


def first_assistant_span(decoded: str) -> str:
    marker = "<|im_start|><|assistant|>:"
    if marker not in decoded:
        return decoded.strip()
    span = decoded.split(marker, 1)[1]
    if "<|im_end|>" in span:
        span = span.split("<|im_end|>", 1)[0]
    return span.strip()


@torch.no_grad()
def generate_replies(model) -> List[Dict]:
    model.eval()
    device = model_device(model)
    rows = []
    for index, messages in enumerate(PROMPTS):
        prompt = assistant_prompt(messages)
        batch = tokenizer(prompt, return_tensors="pt")
        batch = {k: v.to(device) for k, v in batch.items()}
        output = model.generate(
            **batch,
            max_new_tokens=MAX_NEW_TOKENS,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
        )
        decoded = tokenizer.decode(output[0], skip_special_tokens=False)
        rows.append({
            "prompt_id": index,
            "prompt": messages[0]["content"],
            "full_decode": decoded,
            "first_assistant_span": first_assistant_span(decoded),
        })
    return rows


def train_one_method(method_key: str) -> Dict:
    cfg = METHODS[method_key]
    print("=" * 72)
    print("Training", cfg["label"], f"({method_key})")
    print(cfg["what_changed"])
    print("=" * 72)

    torch.cuda.empty_cache()
    model = build_model(cfg)
    device = model_device(model)
    trainable = count_trainable_parameters(model)
    frozen_mb = estimate_frozen_base_memory_mb(model, cfg["load_in_4bit"])

    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
    )
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps,
    )

    history = []
    start = time.time()
    model.train()
    for epoch in range(EPOCHS):
        running = 0.0
        for batch in tqdm(train_loader, desc=f"{method_key} epoch {epoch + 1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            running += float(loss.item())
        epoch_loss = running / len(train_loader)
        history.append({"method": method_key, "epoch": epoch + 1, "train_loss": epoch_loss})
        print(f"epoch {epoch + 1}: train loss {epoch_loss:.4f}")

    train_seconds = time.time() - start
    test_loss = validation_loss(model)
    generations = generate_replies(model)

    adapter_dir = DRIVE_ROOT / "adapters" / method_key
    adapter_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    gen_path = DRIVE_ROOT / "generations" / f"{method_key}.json"
    gen_path.write_text(json.dumps(generations, indent=2))

    del model
    del optimizer
    torch.cuda.empty_cache()

    return {
        "method": method_key,
        "label": cfg["label"],
        "what_changed": cfg["what_changed"],
        "rank": cfg["r"],
        "alpha": cfg["alpha"],
        "use_rslora": cfg["use_rslora"],
        "load_in_4bit": cfg["load_in_4bit"],
        "target_modules": ",".join(cfg["target_modules"]),
        "trainable_params": trainable,
        "frozen_base_memory_mb": round(frozen_mb, 1),
        "updates_per_epoch": len(train_loader),
        "train_seconds": round(train_seconds, 1),
        "test_loss": test_loss,
        "prompts_total": len(PROMPTS),
        "history": history,
        "generations_path": str(gen_path),
        "adapter_dir": str(adapter_dir),
    }

print("Helpers ready.")


## 3. Train and evaluate all four methods

Each method starts from a fresh SmolLM2-135M checkpoint, trains for 3 epochs, writes its adapter and generations to Drive, then frees GPU memory for the next run.


In [ ]:
results = []
all_histories = []

for method_key in ["original_lora", "qlora", "all_linear", "rslora"]:
    summary = train_one_method(method_key)
    all_histories.extend(summary.pop("history"))
    results.append(summary)
    # Checkpoint after every method so a disconnect still leaves partial results.
    pd.DataFrame(results).to_csv(DRIVE_ROOT / "metrics_partial.csv", index=False)
    pd.DataFrame(all_histories).to_csv(DRIVE_ROOT / "train_curves_partial.csv", index=False)
    print("Saved partial metrics for", method_key)

metrics_df = pd.DataFrame(results)
curves_df = pd.DataFrame(all_histories)
metrics_df.to_csv(DRIVE_ROOT / "metrics.csv", index=False)
curves_df.to_csv(DRIVE_ROOT / "train_curves.csv", index=False)
(DRIVE_ROOT / "run_config.json").write_text(json.dumps({
    "model": MODEL_NAME,
    "dataset": f"{DATASET_NAME}/{DATASET_CONFIG}",
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "methods": METHODS,
    "seed": 42,
}, indent=2))

print(metrics_df[[
    "method", "trainable_params", "frozen_base_memory_mb", "test_loss", "train_seconds"
]])


## 4. Collect every prompt reply into one table

The `addressed` column is left blank on purpose. Fill it later by hand with the same rule as LoRATune: does the first assistant span do what was asked, or does it copy / loop / answer something else?


In [ ]:
generation_rows = []
for method_key in METHODS:
    path = DRIVE_ROOT / "generations" / f"{method_key}.json"
    for row in json.loads(path.read_text()):
        generation_rows.append({
            "method": method_key,
            "prompt_id": row["prompt_id"],
            "prompt": row["prompt"],
            "first_assistant_span": row["first_assistant_span"],
            "addressed": "",  # fill later: 1 if on topic, 0 otherwise
        })

generations_df = pd.DataFrame(generation_rows)
generations_df.to_csv(DRIVE_ROOT / "generations.csv", index=False)
print("Wrote", DRIVE_ROOT / "generations.csv")
generations_df.head(4)


## 5. Graphs

Saved under `figures/` on Drive. These are the plots the local report will reuse.


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
FIG = DRIVE_ROOT / "figures"
colors = {
    "original_lora": "#475569",
    "qlora": "#0F766E",
    "all_linear": "#9A3412",
    "rslora": "#1D4ED8",
}
labels = {k: v["label"] for k, v in METHODS.items()}

# 1) Test loss bars
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(metrics_df))
bars = ax.bar(
    x,
    metrics_df["test_loss"],
    color=[colors[m] for m in metrics_df["method"]],
    edgecolor="white",
    width=0.72,
)
ax.axhline(0.883, color="#B45309", linestyle="--", linewidth=1.4, label="Full fine-tune (recorded)")
ax.axhline(1.045, color="#64748B", linestyle=":", linewidth=1.4, label="Homework LoRA (recorded)")
ax.set_xticks(x)
ax.set_xticklabels([labels[m] for m in metrics_df["method"]], rotation=15, ha="right")
ax.set_ylabel("Mean test loss")
ax.set_title("Held-out next-token loss across four LoRA recipes")
ax.legend(frameon=False)
for bar, value in zip(bars, metrics_df["test_loss"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.3f}",
            ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(FIG / "01_test_loss.png", dpi=180)
plt.show()

# 2) Trainable parameters vs test loss
fig, ax = plt.subplots(figsize=(7, 4.5))
for _, row in metrics_df.iterrows():
    ax.scatter(
        row["trainable_params"] / 1e3,
        row["test_loss"],
        s=160,
        color=colors[row["method"]],
        label=labels[row["method"]],
        zorder=3,
    )
    ax.annotate(
        labels[row["method"]],
        (row["trainable_params"] / 1e3, row["test_loss"]),
        textcoords="offset points",
        xytext=(8, 6),
        fontsize=9,
    )
ax.set_xlabel("Trainable parameters (thousands)")
ax.set_ylabel("Mean test loss")
ax.set_title("Quality against adapter size")
ax.legend(frameon=False, loc="best")
fig.tight_layout()
fig.savefig(FIG / "02_params_vs_loss.png", dpi=180)
plt.show()

# 3) Training curves
fig, ax = plt.subplots(figsize=(8, 4.5))
for method_key, group in curves_df.groupby("method"):
    ax.plot(
        group["epoch"],
        group["train_loss"],
        marker="o",
        linewidth=2.2,
        color=colors[method_key],
        label=labels[method_key],
    )
ax.set_xticks([1, 2, 3])
ax.set_xlabel("Epoch")
ax.set_ylabel("Average training loss")
ax.set_title("Training loss by epoch")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / "03_train_curves.png", dpi=180)
plt.show()

# 4) Frozen base memory
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(
    [labels[m] for m in metrics_df["method"]],
    metrics_df["frozen_base_memory_mb"],
    color=[colors[m] for m in metrics_df["method"]],
    edgecolor="white",
)
ax.set_ylabel("Approx. frozen base weight memory (MB)")
ax.set_title("Why QLoRA exists: storage of W, not the adapter")
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
fig.savefig(FIG / "04_frozen_memory.png", dpi=180)
plt.show()

# 5) Pairwise test-loss gaps for the three planned readings
pairs = [
    ("original_lora", "qlora", "Quantization\n(original vs QLoRA)"),
    ("original_lora", "all_linear", "Recipe\n(attention r=8 vs all-linear r=16)"),
    ("all_linear", "rslora", "Scale\n(alpha/r vs alpha/sqrt(r))"),
]
loss_map = dict(zip(metrics_df["method"], metrics_df["test_loss"]))
fig, ax = plt.subplots(figsize=(8, 4.5))
pair_names = [p[2] for p in pairs]
deltas = [loss_map[b] - loss_map[a] for a, b, _ in pairs]
bar_colors = ["#0F766E" if d <= 0 else "#9A3412" for d in deltas]
ax.bar(pair_names, deltas, color=bar_colors, edgecolor="white")
ax.axhline(0, color="#334155", linewidth=1)
ax.set_ylabel("Δ test loss (right minus left)")
ax.set_title("Three controlled comparisons")
for i, d in enumerate(deltas):
    ax.text(i, d + (0.005 if d >= 0 else -0.01), f"{d:+.3f}", ha="center",
            va="bottom" if d >= 0 else "top", fontsize=9)
fig.tight_layout()
fig.savefig(FIG / "05_pairwise_deltas.png", dpi=180)
plt.show()

# 6) Wall-clock cost
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(
    [labels[m] for m in metrics_df["method"]],
    metrics_df["train_seconds"] / 60.0,
    color=[colors[m] for m in metrics_df["method"]],
    edgecolor="white",
)
ax.set_xlabel("Training time (minutes)")
ax.set_title("Wall-clock training cost on this GPU")
fig.tight_layout()
fig.savefig(FIG / "06_train_time.png", dpi=180)
plt.show()

print("Figures saved to", FIG)


## 6. Pack a zip for download

Copy `four_methods_bundle.zip` out of Drive when the run is done. Locally, unpack it under `Week10/loratune/results/four_methods/` and tell the agent training succeeded.


In [ ]:
import shutil

bundle_dir = DRIVE_ROOT
zip_path = Path("/content/drive/MyDrive/loratune_experiments/four_methods_bundle")
shutil.make_archive(str(zip_path), "zip", root_dir=bundle_dir)
print("Bundle:", str(zip_path) + ".zip")
print("Contents:")
for path in sorted(DRIVE_ROOT.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(DRIVE_ROOT), f"({path.stat().st_size} bytes)")


## Done

You should now have on Drive:

- `metrics.csv` — one row per method (test loss, trainable count, memory, timing)
- `train_curves.csv` — epoch losses
- `generations.csv` / `generations/*.json` — all ten prompts × four methods
- `figures/*.png` — six comparison plots
- `adapters/<method>/` — PEFT adapters
- `run_config.json` — exact settings

Fill the blank `addressed` column in `generations.csv` if you want prompt scores in the report. Then download the folder or the zip and continue locally.
